# Notebook 2 · Bloco 2 — O Duto Incremental e o Agente Auditor

**120 minutos: 60 de conceito, 60 de prática.**

O Bloco 1 terminou com a fundação montada a partir de arquivos parados. Dado real não fica parado.
Neste bloco a fonte da Quantum muda embaixo dos seus pés nove vezes, e uma cliente pede para ser
esquecida.

A pergunta do bloco: **quanto do seu duto sobrevive à mudança, e como você prova isso antes de deixar
o agente responder?**

## Passo 1 — preparar a sessão e a fundação do Bloco 1

In [ ]:
# Passo 1 de 10 — preparar a sessão (roda uma vez, ~90 s)
!pip -q install deltalake duckdb sentence-transformers pyyaml pyarrow pandas 2>&1 | tail -1

import os, sys, json, shutil
from pathlib import Path

# O kit vem de um zip. Troque KIT_URL pelo endereço que o professor passar,
# ou faça upload de dutos-do-q.zip no painel de arquivos do Colab (ícone de pasta à esquerda).
KIT_URL = os.environ.get("KIT_URL", "")
RAIZ = Path("/content") if Path("/content").exists() else Path.cwd()
KIT = RAIZ / "dutos-do-q"

if not KIT.exists():
    zip_local = RAIZ / "dutos-do-q.zip"
    if KIT_URL and not zip_local.exists():
        !wget -q -O {zip_local} {KIT_URL}
    assert zip_local.exists(), "Faça upload de dutos-do-q.zip no painel de arquivos, ou preencha KIT_URL."
    shutil.unpack_archive(str(zip_local), str(RAIZ))

sys.path.insert(0, str(KIT / "kit"))
os.chdir(KIT)
print("kit em", KIT)
print(sorted(p.name for p in (KIT / "kit").iterdir()))

from lake import Lake
from contrato import carregar_contratos
import dutos, fundacao, agentes, avaliacao, referencia, aula
from agente import Memoria, CacheSemantico, consumir_eventos

ESQUADRAO = "esquadrao_00"   # <<< TROQUE pelo nome ou número do seu esquadrão
contratos = carregar_contratos()
lake = Lake(str(KIT / "lakehouse_bloco2"))
lake.zerar().criar_todas()

INBOX = dutos.preparar_inbox(KIT)
dutos.bronze(lake, INBOX)
referencia.pipeline(lake, {"fontes": contratos})
dutos.gold(lake)
print(lake.resumo().to_string(index=False))

### A missão, por escrito

In [ ]:
aula.briefing(KIT, "Missão 2")

Repare que o Bloco 2 parte de uma fundação **de referência**, não da que o esquadrão construiu. Isso é
proposital: um erro do Bloco 1 não deve contaminar a nota do Bloco 2, e todo mundo começa do mesmo ponto.

## Passo 2 — os órgãos do Q que vivem fora do lakehouse

A memória de cada cliente e o cache de respostas não são tabelas do lake. Guardem essa frase: ela é a
razão de o caso da Marina não se resolver com um `DELETE`.

In [ ]:
import json
memoria = Memoria(str(KIT / "estado_agente"))
cache = CacheSemantico(str(KIT / "estado_agente"))
memoria.loja.limpar(); cache.loja.limpar()

for m in json.loads((KIT / "dados" / "memorias_seed.json").read_text()):
    memoria.lembrar(m["cliente_id"], m["texto"], m["tipo"], m["data"])
cache.guardar("Qual a tarifa de saque?", "R$ 7,50")

print("memórias da Marina (C0007):", memoria.quantas("C0007"))
for m in memoria.recuperar("C0007", "o que a Marina falou?"):
    print("  ", m["texto"])
print("cache de tarifa:", cache.itens_do_topico("tarifa"))

## Passo 3 — a fonte mutável

O "sistema de origem" da Quantum: três tabelas com Change Data Feed ligado. Elas vão mudar nove vezes
ao longo do bloco.

In [ ]:
print(dutos.semear_fonte(lake))
linha_do_tempo = json.loads((KIT / "dados" / "linha_do_tempo.json").read_text())
for e in linha_do_tempo:
    print(f"  E{e['evento']}  {e['op']:<18} {e['nome']}")

## Passo 4 — o Change Data Feed, cru

Antes de usar, veja o que ele é. A célula abaixo faz uma mudança e lê o feed. Repare no `_change_type`:
um UPDATE aparece como `update_preimage` mais `update_postimage`, ou seja, o estado antes e depois.
É isso que permite saber **o que mudou de fato**, e não apenas que algo mudou.

In [ ]:
v0 = lake.versao("fonte.comunicados")
dutos.avancar_fonte(lake, 1, linha_do_tempo)
dutos.avancar_fonte(lake, 2, linha_do_tempo)
print(lake.mudancas("fonte.comunicados", v0 + 1)[["doc_id","versao","_change_type","_commit_version"]].to_string(index=False))

### Por que o cursor é a versão da tabela, e não o `updated_at`

Um duto que pergunta "o que mudou desde as 14h32" depende do relógio da origem estar certo, de ninguém
gravar com timestamp antigo e de nenhuma linha chegar atrasada. Um duto que pergunta "o que mudou desde
a versão 7" depende do log de transações, que é controlado pelo destino.

Os eventos 7 e 8 existem para provar isso: um reenvio idêntico e uma regravação completa criam versões
novas **sem nenhum dado ter mudado**. Um duto que reage a versões novas sem comparar o efeito líquido
vai re-embedar o corpus inteiro por nada.

## Passo 5 — o ciclo completo: 8 eventos

A cada evento: a fonte muda, o duto incremental consome o CDF, e o agente consome o outbox.
Observem a coluna de embeddings. **Só duas mudanças de texto acontecem em oito eventos.**

In [ ]:
lake.zerar("gold.cursor_sync")
dutos.semear_fonte(lake)
dutos.sync(lake)
consumir_eventos(lake, memoria, cache)

print(f"{'ev':<4} {'o que aconteceu':<48} {'embeds':>7} {'evitados':>9}  outbox")
total = 0
for ev in range(1, 9):
    info = dutos.avancar_fonte(lake, ev, linha_do_tempo)
    s = dutos.sync(lake)
    aplicados = consumir_eventos(lake, memoria, cache)
    total += s["embeds_executados"]
    print(f"E{ev:<3} {info['nome']:<48} {s['embeds_executados']:>7} {s['embeds_evitados']:>9}  {[a['tipo'] for a in aplicados]}")
print("\nembeddings executados no ciclo inteiro:", total)

## Passo 6 — o caso Marina: LGPD não é um DELETE

O evento 6 foi um pedido de eliminação. O duto apagou as transações e pseudonimizou o cadastro. Mas a
memória da Marina vive num arquivo JSON ao lado do agente, e o cache pode ter guardado uma resposta que
cita o nome dela. O lakehouse não alcança nenhum dos dois.

Por isso o duto **publica um evento** em vez de tentar apagar: `esquecer_cliente` no outbox. Quem vive
fora do lake consome e executa. Se ninguém consumir, o dado sobrevive, a auditoria passa, e a empresa
descobre no pedido de informação do titular.

In [ ]:
print("transações da Marina na Silver:", lake.escalar("SELECT COUNT(*) FROM silver.transacoes WHERE cliente_id='C0007'"))
print("cadastro:", lake.sql("SELECT cliente_id, nome, cpf, cidade FROM silver.clientes WHERE cliente_id='C0007'").to_string(index=False))
print("memórias da Marina:", memoria.quantas("C0007"))
print()
print(lake.sql("SELECT tipo, payload, consumido FROM gold.eventos_agente ORDER BY criado_em").to_string(index=False))

## Passo 7 — as views que o Q enxerga

O Q não escreve SQL. Ele chama consultas parametrizadas sobre views aprovadas. A regra de negócio mora
na view, não no prompt: quando o comitê mudar o percentual do CDB, muda a view, e nenhum agente precisa
ser reeducado.

In [ ]:
print(dutos.bcb(lake, KIT / "dados" / "bcb" / "snapshot.json"))
print(dutos.views(lake))

In [ ]:
# a mesma pergunta, três datas: dia útil, sábado e feriado. O carry-forward acontece na CONSULTA.
for data in ("2026-09-11", "2026-09-12", "2026-09-07"):
    r = lake.sql("""SELECT data, valor FROM gold.v_indicadores
                    WHERE serie='4389' AND data <= CAST($d AS DATE) ORDER BY data DESC LIMIT 1""", {"d": data})
    print(f"  CDI vigente em {data}: {r.iloc[0].to_dict() if len(r) else 'sem dado'}")

## Passo 8 — DECISÃO DO ARQUITETO · as regras do Auditor

Agora a parte que vale ponto neste bloco. O Agent 2 é um LLM-as-a-judge: ele recebe **evidências
determinísticas** coletadas do lakehouse e as **regras que vocês escreverem**, e decide se os dutos
podem conectar ao Q.

O LLM julga. Ele não mede. Essa separação é o que torna o Auditor auditável.

Perguntas para o esquadrão decidir:

- Qual taxa de rejeição é aceitável? 5% é pouco ou muito para uma fintech?
- O que é bloqueante e o que é aviso? Um indicador de mercado com 27 dias derruba o agente inteiro?
- O que a LGPD exige que esteja em zero, sem margem?
- Um chunk sem embedding é erro ou é normal?

In [ ]:
evidencias = agentes.coletar_evidencias(lake, memoria=memoria)
print(json.dumps(evidencias, indent=1, ensure_ascii=False))

In [ ]:
# As regras que vocês receberam. Rodem a célula e leiam com atenção antes de mudar:
# elas aprovam uma fundação com a Marina viva no sistema e com o indicador parado há 27 dias.
regras = agentes.REGRAS_INICIAIS
print(regras)

Endureçam as regras abaixo. Cada regra precisa citar o **nome exato** de um campo das evidências que
vocês acabaram de ver, porque o filtro anti-alucinação descarta violação que não aponta campo real.

In [ ]:
regras = """
1. ESCREVAM AQUI. Uma regra por linha, citando o campo das evidências e o limiar.
   Exemplo de forma: "Taxa de rejeição abaixo de X% (taxa_rejeicao_pct)."
2.
3.
4.
5.
6.
"""
print(regras)

## Passo 9 — o Auditor julga, a catraca grava

In [ ]:
if agentes._backend() == "mock":
    print(agentes.carregar_modelo("Qwen/Qwen2.5-Coder-1.5B-Instruct"))

veredito = agentes.julgar(regras, evidencias)
print(json.dumps({k: veredito[k] for k in ("decisao","violacoes","violacoes_descartadas","justificativa")},
                 indent=1, ensure_ascii=False))
print("\nliberação gravada:", agentes.liberar(lake, veredito))

Repare em `violacoes_descartadas`. Um LLM que julga números inventa números. O filtro exige que cada
violação aponte um campo que existe nas evidências, com o valor que está lá. O que não aponta é
descartado antes de virar decisão.

Sem esse filtro, o Auditor bloqueia a produção por um problema que não existe, e alguém desliga o
Auditor na segunda vez que isso acontece.

## Passo 10 — o harness do Auditor

Quatro cenários com evidências controladas: fundação limpa, Marina viva, dado velho e índice
contaminado. O Auditor precisa acertar a decisão **e** apontar as evidências certas.

In [ ]:
resultado = avaliacao.avaliar_auditor(regras)
print("\narquivo salvo em:", avaliacao.salvar(resultado, str(KIT / "resultados")))

## Passo 11 — corrida de freshness (o professor anuncia)

Evento 9: a Quantum publica um comunicado novo. O cronômetro começa quando o professor avisa e para
quando o Q responde com a informação correta. O que está sendo medido é o caminho inteiro, da mudança
na fonte até a resposta liberada.

In [ ]:
import time
t0 = time.time()

dutos.avancar_fonte(lake, 9, linha_do_tempo)
dutos.sync(lake)
consumir_eventos(lake, memoria, cache)

veredito = agentes.julgar(regras, agentes.coletar_evidencias(lake, memoria=memoria))
agentes.liberar(lake, veredito)

if agentes.status_liberacao(lake) == "PASS":
    topo = dutos.buscar(lake, "Até que horas vai o atendimento telefônico?", k=1)
    resposta = topo.iloc[0]["texto"] if len(topo) else "(nada no índice)"
else:
    resposta = f"o Q está bloqueado pela catraca: {veredito['justificativa']}"

print(f"{round(time.time() - t0, 1)}s · liberação = {agentes.status_liberacao(lake)}")
print(resposta[:400])

## Passo 12 — responder e entregar

Mesma coisa do bloco anterior: escrevam entre as aspas e rodem a célula. Ela grava
`entrega_<esquadrao>_bloco2.json` com o score do Auditor, as regras que vocês escreveram e as respostas.
Depois, baixem o notebook em **Arquivo > Fazer download > Fazer download do .ipynb**.

In [ ]:
respostas = {

"1. Qual regra do Auditor vocês endureceram, e por que esse limiar e não outro? "
"Qual regra vocês deixaram como aviso em vez de bloqueio, e por quê?":
"""

""",

"2. Na revogação do comunicado, apagar de verdade ou marcar como inativo? "
"O que cada opção custa quando alguém pede o histórico seis meses depois, e a resposta muda "
"se o que foi revogado for um dado pessoal sob pedido de eliminação?":
"""

""",

"3. Se o consumidor do outbox parasse de rodar por uma semana, o que quebraria primeiro? "
"Como vocês descobririam sem um cliente ligar: qual métrica, coletada onde, com qual limiar?":
"""

""",

}

avaliacao.gerar_entrega(
    esquadrao=ESQUADRAO, bloco=2, caminho_kit=KIT,
    resultados={"auditor": resultado},
    decisoes=respostas,
    regras_auditor=regras,
)

### O fio da aula

Três agentes, uma tese: um ecossistema agêntico é tão bom quanto o frescor, a qualidade e a governança
da fundação de dados que o sustenta. O Q não ficou mais inteligente hoje. Ficou confiável, e isso foi
decidido no contrato, no cursor e na catraca.